## CNN for MNIST

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST

In [ ]:
# Define the CNN architecture
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.fc_layer = nn.Sequential(
            nn.Linear(64 * 7 * 7, 1024),
            nn.ReLU(),
            nn.Linear(1024, 10)
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x

In [ ]:
# Setup training parameters
batch_size = 64
learning_rate = 0.001
epochs = 5

In [ ]:
# MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Determine the device and move the model to that device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Training the model
for epoch in range(epochs):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f'Epoch {epoch+1}/{epochs} completed.')

Epoch 1/5 completed.
Epoch 2/5 completed.
Epoch 3/5 completed.
Epoch 4/5 completed.
Epoch 5/5 completed.


In [ ]:
# Evaluating the model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = correct / total
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

Test Accuracy: 99.23%


## ViT for MNIST

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch import optim

class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert self.head_dim * heads == embed_size, "Embed size needs to be divisible by heads"

        # Ensure these linear transformations project to the correct dimensions
        self.values = nn.Linear(embed_size, embed_size, bias=False)
        self.keys = nn.Linear(embed_size, embed_size, bias=False)
        self.queries = nn.Linear(embed_size, embed_size, bias=False)
        self.fc_out = nn.Linear(embed_size, embed_size)


    def forward(self, value, key, query):
        N = query.shape[0]
        value_len, key_len, query_len = value.shape[1], key.shape[1], query.shape[1]

        values = self.values(value).view(N, value_len, self.heads, self.head_dim)
        keys = self.keys(key).view(N, key_len, self.heads, self.head_dim)
        queries = self.queries(query).view(N, query_len, self.heads, self.head_dim)

        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])

        attention = torch.softmax(energy / (self.embed_size ** (1 / 2)), dim=3)

        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )

        return self.fc_out(out)

class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super(TransformerBlock, self).__init__()
        self.attention = SelfAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query):
        attention = self.attention(value, key, query)
        x = self.dropout(self.norm1(attention + query))
        forward = self.feed_forward(x)
        out = self.dropout(self.norm2(forward + x))
        return out

class ViT(nn.Module):
    def __init__(self,
                 image_size,
                 patch_size,
                 num_classes,
                 channels,
                 depth,
                 heads,
                 embed_size,
                 forward_expansion,
                 dropout):

        super(ViT, self).__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.embed_size = embed_size
        self.channels = channels
        self.projection = nn.Linear(patch_size * patch_size * channels, embed_size)
        self.position_embeddings = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_size))
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_size))
        self.dropout = nn.Dropout(dropout)

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(embed_size, heads, dropout, forward_expansion) for _ in range(depth)
        ])

        self.to_cls_token = nn.Identity()

        self.fc = nn.Linear(embed_size, num_classes)

    def forward(self, x):
        N, _, _, _ = x.shape
        x = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
        x = x.contiguous().view(N, self.num_patches, -1)
        x = self.projection(x)
        cls_tokens = self.cls_token.expand(N, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.position_embeddings
        x = self.dropout(x)

        for block in self.transformer_blocks:
            x = block(x, x, x)

        cls_token = self.to_cls_token(x[:, 0])
        return self.fc(cls_token)

# Load and preprocess MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize model, loss, and optimizer
model = ViT(
    image_size=28,
    patch_size=7,
    num_classes=10,
    channels=1,
    depth=6,
    heads=8,
    embed_size=64,
    forward_expansion=4,
    dropout=0.1,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training and evaluation functions
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def test(dataloader, model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            total += y.size(0)
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Execute training and testing
for epoch in range(10):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
    test(test_loader, model)

print("Done!")

Epoch 1
-------------------------------
Test Accuracy: 91.18%
Epoch 2
-------------------------------
Test Accuracy: 94.80%
Epoch 3
-------------------------------
Test Accuracy: 95.95%
Epoch 4
-------------------------------
Test Accuracy: 96.72%
Epoch 5
-------------------------------
Test Accuracy: 96.23%
Epoch 6
-------------------------------
Test Accuracy: 96.63%
Epoch 7
-------------------------------
Test Accuracy: 96.93%
Epoch 8
-------------------------------
Test Accuracy: 97.06%
Epoch 9
-------------------------------
Test Accuracy: 97.21%
Epoch 10
-------------------------------
Test Accuracy: 97.10%
Done!


Achieving 99% accuracy with a Convolutional Neural Network (CNN) and 97% with a Vision Transformer (ViT) on the MNIST dataset presents an interesting comparison. The difference in performance and the choice between CNNs and Transformers for image classification can be analyzed from several angles:

### 1. **Model Architecture:**

- **CNNs** are specifically designed for image data, leveraging spatial hierarchies and local connectivity through convolutional filters. This allows them to be highly efficient and effective for image classification tasks, particularly on datasets like MNIST where spatial patterns (such as edges and curves) are key to distinguishing between classes.
- **ViTs**, on the other hand, do not inherently process spatial information but rather learn to do so through self-attention mechanisms across patches of images. While they have shown remarkable success on various tasks, especially with larger and more complex datasets, their performance on simpler, spatially structured tasks like MNIST might not always surpass that of CNNs, particularly with a straightforward implementation and without extensive tuning.

### 2. **Data Complexity and Model Capacity:**

- The MNIST dataset, while a benchmark for classification models, is relatively simple in terms of the complexity of its images (handwritten digits on a uniform background). CNNs can quickly learn the spatial hierarchies present in such images with fewer parameters and less computational cost.
- ViTs excel in environments where the dataset is large and complex enough to justify their increased model complexity and capacity. Their performance advantage over CNNs becomes more pronounced in scenarios where the dataset contains a lot of variability and the task benefits from understanding long-range dependencies within the image.

### 3. **Training Data Size:**

- Transformers generally require more data to train effectively compared to CNNs because of their larger number of parameters and their reliance on self-attention to learn spatial relationships. The relatively small size of the MNIST dataset (60,000 training images) might not be sufficient to fully leverage the ViT’s capabilities.
- CNNs, with their localized receptive fields and shared weights, are more data-efficient for learning from images, making them particularly well-suited for tasks like MNIST.

### 4. **Generalization and Overfitting:**

- The slight drop in accuracy with the ViT could also be attributed to overfitting, given that Transformers have a large number of parameters. Without proper regularization or if the model complexity is too high relative to the task, the model might learn noise in the training data, leading to worse generalization.
- CNNs, with their built-in inductive biases towards spatial data, might generalize better from the same amount of training data, leading to higher test accuracy on MNIST.

### Conclusion:

The difference in performance highlights the importance of choosing the right tool for the job. While ViTs have shown impressive results on complex image classification tasks and large datasets, CNNs still hold a strong position for tasks involving spatial data, especially when the dataset is relatively simple or small. This comparison also underscores the significance of model complexity, data efficiency, and the inherent characteristics of the dataset in choosing between a CNN and a ViT. For future work, experimenting with hybrid models or more sophisticated versions of ViTs could potentially close the gap in performance on tasks like MNIST, while also providing insights into the trade-offs between these two powerful architectures.